In [0]:
# 03c_apply_changes：把 Bronze 变更表回放到 Silver（幂等，可重复运行）
from pyspark.sql import functions as F, Window

TARGET        = "nyc_taxi.silver.yellow_trips"
CHANGES_TABLE = "nyc_taxi.bronze.yellow_trip_changes"

target_cols = spark.table(TARGET).columns

# 1) 每个 trip_id 只保留最新的变更
w = Window.partitionBy("trip_id").orderBy(F.col("_change_ts").desc())
src_latest = (spark.table(CHANGES_TABLE)
    .withColumn("_source_file", F.lit(CHANGES_TABLE))
    .select(*target_cols, "_change_type", "_change_ts")
    .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn"))

n_src = src_latest.count()
print("待回放变更数:", n_src)
if n_src == 0:
    dbutils.notebook.exit("no changes")   # 没有变更就直接结束，不产生空提交

src_latest.createOrReplaceTempView("changes_latest")

In [0]:
# 2) 生成 MERGE：带 null 安全的变化检测
non_key      = [c for c in target_cols if c != "trip_id"]
compare_cols = [c for c in non_key if not c.startswith("_")]
set_clause   = ",\n  ".join(f"t.`{c}` = s.`{c}`" for c in non_key)
insert_cols  = ", ".join(f"`{c}`" for c in target_cols)
insert_vals  = ", ".join(f"s.`{c}`" for c in target_cols)
CHANGED      = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in compare_cols)

merge_sql = f"""
MERGE INTO {TARGET} AS t
USING changes_latest AS s
ON t.trip_id = s.trip_id
WHEN MATCHED AND s._change_type = 'DELETE' THEN DELETE
WHEN MATCHED AND ({CHANGED}) THEN UPDATE SET
  {set_clause}
WHEN NOT MATCHED AND s._change_type != 'DELETE' THEN
  INSERT ({insert_cols}) VALUES ({insert_vals})
"""
display(spark.sql(merge_sql))

In [0]:
# 3) 门禁：MERGE 后 trip_id 仍然唯一
s = spark.table(TARGET)
total, uniq = s.count(), s.select("trip_id").distinct().count()
print(f"Silver 行数 {total:,} / trip_id 唯一数 {uniq:,}")
assert total == uniq, "MERGE 后 trip_id 不唯一！"